In [12]:
# !pip install matplotlib
# pip install --upgrade pip
# !pip uninstall numpy --yes
# !pip install "numpy<2"
# !pip install --upgrade numexpr bottleneck
# !pip install lets_plot
# !pip install scikit-learn

In [13]:
import numpy as np
import pandas as pd

import matplotlib

In [2]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib as mpl

from pandas_datareader import data as pdr

import datetime as dt
import yfinance as yf

# Download data

In [4]:
# TODO yf download
def download_yf_data(tickers, start_date, end_date, column='Close'):
    """
    Download AAPL stock data from Yahoo Finance
    download_yf_data(['AAPL'], '2020-01-01', '2021-01-01', ['Close'])
    """
    stock_data = yf.download(
        tickers,
        start=start_date,
        end=end_date,
        threads=True
    )
    return stock_data[column]

# Example usage
# stock_data = download_yf_data(['AAPL', 'MSFT'], '2020-01-01', '2021-01-01', ['Close'])
# stock_data = stock_data.droplevel('Price', axis=1)

In [ ]:
# TODO csv download
def download_csv_data(file_path):
    """
    Download stock data from a CSV file
    """
    stock_data = pd.read_csv(file_path, 
                             index_col=0, 
                             parse_dates=True)
    return stock_data

# stock_data = download_csv_data('alphas.csv')


## TODO Read multi-index
# stock_data = pd.read_csv('stock_data_2.csv', 
#                         index_col=0, 
#                         parse_dates=True,
#                         header=[0,1]
#                         )

# TODO slice multi-index to get 'Adj Close' prices only
# stock_data = stock_data.xs('Adj Close', level=1, axis=1


# Clean missing cols

In [ ]:
def missing_cols_remove(stock_data, threshold=.3):
    """
    remove cols with missing values >  30%
    """
    display(stock_data.isnull().sum())
    missing = stock_data.isnull().mean().sort_values()
    print("Drop cols")
    drop_list = missing[missing>threshold].index.to_list()
    stock_data =\
        stock_data.drop(labels=drop_list, axis=1)
    print("ffill na")
    print("drop na for any trailing NA vals")
    stock_data = stock_data.fillna(method="ffill").dropna()

    return stock_data

# missing_cols_remove(stock_data)

# Returns 

In [ ]:
# once positions are created, need to get trade, cum rets, drawdown
"""
TODO: LOG RETURNS get_returns
"""
def get_returns_log(df, position_shift):
    """
    Log returns, given a list of positions and stock price in first column
    Usage: get_returns_log(aapl, 0)
    """
    df['positions'] = df['positions'].shift(position_shift)      # IMPORTANT
    df['trade'] = \
    (
        df['positions'].diff().fillna(0)
    )
    if df.at[df.index[0], 'positions'] != 0:
        df.at[df.index[0], 'trade'] = df.at[df.index[0], 'positions']
    df['passive_returns'] = \
    (
        np.log(
            df[df.columns[0]] / 
                df[df.columns[0]].shift(1)
        )
    ).fillna(0)
    df['strategy_returns'] = \
    (
        df['passive_returns'] * df['positions'].fillna(0)
    )
    df['cum_passive_returns'] = \
    (
        df['passive_returns'].cumsum().apply(np.exp).fillna(1)
    )
    df['cum_strategy_returns'] = \
    (
        df['strategy_returns'].cumsum().apply(np.exp).fillna(1)
    )
    

    return df

"""
TODO: PCT RETURNS get_pct_returns
"""
def get_returns_simple(df, position_shift):
    """
    Log returns, given a list of positions and stock price in first column
    Usage: get_returns_simple(aapl, 0)
    """
    df['positions'] = df['positions'].shift(position_shift) # IMPORTANT
    df['trade'] = \
    (
        df['positions'].diff().fillna(0)
    )
    if df.at[df.index[0], 'positions'] != 0:
        df.at[df.index[0], 'trade'] = df.at[df.index[0], 'positions']
    df['passive_returns'] = \
    (
        df[df.columns[0]].pct_change()
    ).fillna(0)
    df['strategy_returns'] = \
    (
        df['passive_returns'] * df['positions'].fillna(0)
    )
    df['passive_gross_returns'] = \
    (
        df['passive_returns'] + 1
    )
    df['strategy_gross_returns'] = \
    (
        df['strategy_returns'] + 1
    )
    df['cum_passive_returns'] = \
    (
        df['passive_gross_returns'].cumprod().fillna(1)
    )
    df['cum_strategy_returns'] = \
    (
        df['strategy_gross_returns'].cumprod().fillna(1)
    )

    return df

# get_daily drawdown
daily_drawdown = \
(
    df['cum_strategy_returns'] / df['strategy_gross_returns'].cummax() - 1
)

In [ ]:
def get_pnl(df, shares):
    df['pnl'] = df['pnl'][df.columns[0]].diff() * shares * df['positions']
    
    return df

In [ ]:
def get_capital(aapl, initial_cap=100_000, if_log=True):
    """
    Calculate cum strategy capital after trading costs
    """
    cap = initial_cap
    cum_strategy_returns = pd.Series(index=aapl.index)

    for idx, row in aapl.iterrows():
        if if_log:
            cap *= np.exp(row['strategy_returns'])
        else:
            cap *= (1 + row['strategy_returns'])
        if row['trade'] !=0:
            cap -= 5
        cum_strategy_returns.loc[idx] = cap

    print(f"Final capital ${cap:,.0f}")

    return cum_strategy_returns

In [ ]:
# TODO: get_annual_returns dataframe
# prepare annual returns
ann_rets = stock_data.pct_change().mean() * 252
ann_vols = stock_data.pct_change().std() * np.sqrt(252)

df = \
(
    pd.DataFrame({'return': ann_rets, 'volatility': ann_vols})
)

## SCALED DATA FOR CLUSTERING
# from sklearn.preprocessing import StandardScaler
# scaler = StandardScaler()
# data = scaler.fit_transform(df)

# df = pd.DataFrame(data, columns=df.columns, index=df.index)

# X = df.copy()

# Ready Functions

In [ ]:
def get_momentum_strategy(df, sma_ls, position_shift=1):
    """
    Apply single stock momentum strategy pipeline to price DataFrame `df`. 
    Price is in first column.
    Adds columns:
      positions, trade, passive_returns, strategy_returns,
      cum_returns, cum_strategy_returns
    Parameters:
      df        : pd.DataFrame with price columns for ticker name as the header
      sma_tuple : tuple of (short_window, long_window) for SMAs
    Returns:
      pd.DataFrame with the above columns
    """
    def get_sma(df, sma_list):
        """
        Take DF and given list of SMA to enrich df when stock price is in first column
        """
        for window in sma_list:
            df[f"sma_{window}"] =\
            (
                df[df.columns[0]]
                .rolling(window = window,
                         min_periods = 1,
                         center = False)
                .mean()
            )

        return df
    if sma_ls[0] >= sma_ls[1]:
        raise ValueError("sma_short must be less than sma_long")
    df = get_sma(df, sma_ls).dropna()

    df['positions'] =\
    (
        np.where((df[f'sma_{sma_ls[0]}'] > df[f'sma_{sma_ls[1]}']), 1, 0)
    )
    df['positions'] = df['positions'].shift(position_shift).fillna(0)
    df['trade'] = \
    (
        df['positions'].diff().fillna(0)
    )
    if df.at[df.index[0], 'positions'] != 0:
        df.at[df.index[0], 'trade'] = df.at[df.index[0], 'positions']
    df['passive_returns'] =\
    (    # get passive returns
        np.log(df[df.columns[0]] 
            / df[df.columns[0]].shift(1))
    ).fillna(0)
    df['strategy_returns'] =\
    (     # get strategy returns
        df['passive_returns'] * df['positions'].shift(1).fillna(0)
    )
    df['cum_returns'] =\
    (
        df['passive_returns'].cumsum().apply(np.exp).fillna(1)
    )
    df['cum_strategy_returns'] =\
    (
        df['strategy_returns'].cumsum().apply(np.exp).fillna(1)
    )

    return df

# sma_list = [42, 252]
# get_momentum_strategy(df[['GS']], sma_list)

In [ ]:
def get_bollinger_strategy(df, 
                              window = 20,
                              num_std = 2):
    """
    Create a Bollinger Band mean-reversion trading strategy
    """
    # get sma
    def get_sma_(df, sma_list, min_periods):
        """
        Take DF and given list of SMA to enrich df when stock price is in first column
        """
        for window in sma_list:
            df[f"sma_{window}"] =\
            (
                df[df.columns[0]]
                .rolling(window = window,
                        min_periods = min_periods,
                        center=False)
                .mean()
            )

        return df
    df = df.copy()
    df = get_sma_(df, [window], window)
    std = df[df.columns[0]].rolling(window).std()
    df = df.dropna()
    # print(df[f'sma_{window}'])
    df['upper'] = df[f'sma_{window}'] + num_std * std
    df['lower'] = df[f'sma_{window}'] - num_std * std

    df['positions'] =\
    (
        np.where(
            df[df.columns[0]] < df['lower'], 1, 
                np.where(
                    df[df.columns[0]] > df['upper'], 0, 
                        np.where(
                            (df[df.columns[0]] < df[f'sma_{window}']) & 
                            (df[df.columns[0]] > df[f'sma_{window}']).shift(1), 0,
                                np.where(
                                    (df[df.columns[0]] > df[f'sma_{window}']) & 
                                    (df[df.columns[0]] < df[f'sma_{window}']).shift(1), 0, np.nan
                                )
                        )
                )
        )
    )
    df['positions'] = df['positions'].ffill().shift(1).fillna(0)
    
    return df

aapl = get_bollinger_strategy(aapl.dropna(), 20, 2)

In [ ]:
# get sma
def get_sma(df, sma_list):
    """
    Take DF and given list of SMA to enrich df when stock price is in first column
    """
    df = df.copy()
    
    for window in sma_list:
        df[f"sma_{window}"] =\
        (
            df[df.columns[0]]
            .rolling(window = window,
                     min_periods = int(window * .8))
            .mean()
        )

    return df

# sma_list = [21, 63, 252]
# df_sma = get_sma(stock_data[[target]].copy(), sma_list)



In [ ]:
# get_EMA
def get_ema(df, ema_list):
    """
    Take DF and given list of EMA to enrich df when stock price is in first column
    """
    df = df.copy()
    
    for window in ema_list:
        df[f"ema_{window}"] =\
        (
            df[df.columns[0]]
            .ewm(span = window)
            .mean()
        )

    return df

In [ ]:
# get_rsi
def get_rsi(df, period_list=[10, 30, 200]):
    """
    Calculate RSI (Relative Strength Index)
    Parameters:
    df: pandas df of closing price
    period_list: RSI period (default [10, 30, 200])
    Returns:
    pandas df with RSI values
    """
    # Make a copy to avoid SettingWithCopyWarning
    df = df.copy()
    
    # Calculate price changes
    delta = df[df.columns[0]].diff()
    
    for period in period_list:
        # Separate gains and losses
        gain = delta.clip(lower=0)
        loss = -delta.clip(upper=0)
        
        # get first average
        avg_gain = pd.Series(index=df.index, dtype='float64')
        avg_loss = pd.Series(index=df.index, dtype='float64')

        # get first average
        avg_gain.iloc[period] = gain.iloc[1:period+1].mean()
        avg_loss.iloc[period] = loss.iloc[1:period+1].mean()

        # Calculate rolling averages
        for i in range(period + 1, len(df)): 
            avg_gain.iloc[i] = (avg_gain.iloc[i-1] * (period - 1) + gain.iloc[i]) / period
            avg_loss.iloc[i] = (avg_loss.iloc[i-1] * (period - 1) + loss.iloc[i]) / period

        # Calculate RS (Relative Strength)
        rs = avg_gain / avg_loss
        
        # Calculate RSI
        rsi = 100 - (100 / (1 + rs))

        df[f'rsi_{period}'] = rsi
        
    return df

In [ ]:
# get_stochastic_oscillator
def get_stochastic_oscillator(ticker, start, end, period_list=[10, 30, 200]):
    """
    Get the stochastic oscillator for a given stock.
    """
    def STOK(close, low, high, n):
        STOK = ((close - low.rolling(n).min()) / \
                (high.rolling(n).max() - low.rolling(n).min()
                )
            ) * 100
        return STOK

    def STOD(close, low, high, n):
        STOK = ((close - low.rolling(n).min()) / \
                (high.rolling(n).max() - low.rolling(n).min()
                )
            ) * 100
        
        STOD = STOK.rolling(3).mean()
        return STOD

    df = yf.download(
                        ticker,
                        start=start,
                        end=end
                    ).droplevel('Ticker',axis=1)

    for period in period_list:
        df[f"%K{period}"] = STOK(df["Close"], df["Low"], df["High"], period)
        df[f"%D{period}"] = STOD(df["Close"], df["Low"], df["High"], period)

    return df.drop(columns=['Open', 'High', 'Low', 'Close', 'Volume'])

# get_stochastic_oscillator('AAPL', '2020-01-01', '2023-01-01')

/var/folders/h2/r7qn2m9n1zb6y_0q191gdqth0000gn/T/ipykernel_29231/347669752.py:22: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(
[*********************100%***********************]  1 of 1 completed


Price,%K10,%D10,%K30,%D30,%K200,%D200
Date,,,,,,
2020-01-02,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-03,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-06,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-07,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-08,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
2022-12-23,10.919816,16.754102,9.269308,14.200844,6.405218,9.037122
2022-12-27,6.164684,9.579264,5.267373,8.136674,2.756393,5.434843
2022-12-28,0.817693,5.967398,0.613269,5.049983,0.321169,3.160927


In [ ]:
# Rate of Change (RoC)
def get_roc(df, period_list):
    """
    Calculate the Rate of Change (RoC) for a given DataFrame and period.
    """
    for period in period_list:
        M = df[df.columns[0]].diff(period - 1)
        N = df[df.columns[0]].shift(period - 1)
        df[f"roc_{period}"] = (M / N) * 100

    return df

# get_roc(stock_data[[target]].copy(), [10, 30]).drop(columns=target)


In [ ]:
"""
TODO get_drawdown_top max drawdown and duration table
"""
from lets_plot import *
LetsPlot.setup_html()

def get_drawdowns_plot(df):
    df['daily_drawdown'] = df['cum_strategy_returns'] / df['cum_strategy_returns'].cummax() - 1
    # take date column out and cumsum for periods
    dd_reset = df.reset_index()
    dd_reset['period'] = (dd_reset['daily_drawdown'] == 0).cumsum()
    # find nodes with 0 value i.e. highpoints
    dd_nonzero = dd_reset[dd_reset['daily_drawdown'] != 0]
    # aggregate by period
    period_stats = dd_nonzero.groupby('period').agg(
        start_date = ('Date', 'min'),
        end_date = ('Date', 'max'),
        avg_dd = ('daily_drawdown', 'mean'),
        max_dd = ('daily_drawdown', 'min'),
        duration=('daily_drawdown', 'count')
    ).sort_values(by='avg_dd',ascending=True)


    ## printing drawdowns
    for i, row in enumerate(period_stats.itertuples(), start=1):
        ordinal = f"{i}{'tsnrhtdd'[i%5*(i%100^15>4>i%10)::4]}"
        print(f"The {ordinal} worst drawdown period : Net drawdown {-row.max_dd:.2%}% | Duration {row.duration} Days\n")


    # plot drawdowns
    # from lets_plot import *
    # LetsPlot.setup_html()
    p =\
    (
        ggplot(dd_reset, aes(x='Date')) +
        geom_line(aes(y='daily_drawdown'), color='grey', size=0.7) +
        # geom_line(aes(y='sma_50'), color='orange', size=0.7,linetype=2) +   # amend sma as required
        # geom_line(aes(y='sma_200'), color='blue', size=0.7,linetype=2) +
        # geom_point(aes(y='sma_50'), data=wmt_melt[wmt_melt['trade']>0.0], color='red', size=3) +       # amend trade as required
        # geom_point(aes(y='sma_50'), data=wmt_melt[wmt_melt['trade']<-0.0], color='blue', size=3) +
        ggtitle(f"Drawdown for {ticker}") +
        ylab("pct loss") +
        scale_y_continuous(format='.0%') + 
        xlab("Date") +
        ggsize(1200, 500) 
    )
    display(p)

    return period_stats


def get_drawdowns(cum_returns):
    """
    Calculate the Maximum Drawdown (MDD) and the Longest Drawdown Duration (LDD)
    from a cumulative returns SERIES.

    Parameters:
    cum_returns : pd.SERIES of cumulative returns
    Returns:
    tuple (mdd, ldd)
    Usage: 
    get_drawdowns(df['cum_strategy_returns'])
    """
    def get_longest_drawdown_period(dd, cum_returns):
        periods =\
        (
            np
            .diff(np.append(dd[dd == 0].index,
                            dd.index[-1: ]
                            )
                )
        )
        return periods.max() / np.timedelta64(1, "D")

    daily_drawdown = cum_returns / cum_returns.cummax() - 1
    mdd = daily_drawdown.min()
    ldd = get_longest_drawdown_period(daily_drawdown, cum_returns) # days
    return mdd, ldd


# get_drawdowns(cum_rets['Alphas A'])


NameError: name 'df' is not defined

In [ ]:
# Above and below average years
splits = [
            '2015-01-01',
            '2016-01-01',
            '2017-01-01',
            '2018-01-01',
]

dct = {}
ar_dct = {}

for idx, year in enumerate([2015, 2016, 2017]):
    dct[year] = aapl.loc[splits[idx]: splits[idx+1]]

for year, df in dct.items():
    cum_rets = df['strategy_returns'].cumsum().apply(np.exp).fillna(1)
    ar_dct[year] = cum_rets.iloc[-1] - 1

avg = (np.mean(list(ar_dct.values())))
above = [k for k, v in ar_dct.items() if v > avg]
below = [k for k, v in ar_dct.items() if v < avg]

f"    Below average year(s) : {below}. Above average year(s) : {above}" 

In [ ]:
def get_sharpe(daily_returns):
    """
    get_sharpe Calculate the annualized Sharpe ratio from a series of daily returns.
    Annualized Sharpe ratio computed as:
                sqrt(252) * mean(daily_returns) / std(daily_returns)
    - Assumes 252 trading days per year for annualization. Adjust the
      multiplier for a different convention.
    - Input should be returns (not prices). Convert prices to returns before
      calling this function.
    Parameters:
        daily_returns : array-like (pd.Series or np.ndarray)
    Returns:
        float
    Usage:
        get_sharpe_ratio(df['strategy_returns'])
    """
    return np.sqrt(252) * daily_returns.mean() / daily_returns.std(ddof = 1)

# get_sharpe_ratio(ibm['strategy_returns'])
# print(f"The answer is {max(sharpe_dct, key=s_dct.get).upper()} with  {max(sharpe_dct.values()):.5f}")

In [ ]:
def get_macd(
    df: pd.DataFrame,
    span: list[int, int, int]
    ) -> pd.DataFrame:
    """
    Compute MACD / get_macd and signal-line for AAPL prices and attach intermediate EWMAs.
    Assumes first col is ticker data

    Parameters
        df : pandas.DataFrame Time-indexed DataFrame containing a column named "AAPL" with price data.
        span : list[int, int, int] Three integers specifying spans for exponential moving averages:
    Returns:
        pandas.DataFrame
    Usage:
        get_MACD(aapl, [12, 26, 9])
    """
    ticker = df.columns[0]
    df[f"_{span[0]}_ewma"] =\
    (
        df
        [df.columns[0]]
        .ewm(span = span[0]
            )
        .mean()
    )

    df[f"_{span[1]}_ewma"] =\
    (
        df
        [df.columns[0]]
        .ewm(span = span[1]
            )
        .mean()
    )

    df["macd"] =\
    (
        df
        .eval(
            f"`_{span[0]}_ewma` - `_{span[1]}_ewma`"
        )
    )

    df["macd_signal"] =\
    (
        df
        ["macd"]
        .ewm(span = span[2]
            )
        .mean()
    )
    return df

# get_MACD(msft, [12, 26, 9])

In [ ]:
def get_cagr(cumulative_returns, annual_days=365.25):
    """
    get_cagr Compute Compound Annual Growth Rate (CAGR) from a series of cumulative returns.
        NOTE: please ensure index is date
    Parameters:
        cumulative_returns : pandas.Series. Time-indexed series of cumulative returns (e.g. cumulative growth factors)
    Returns:
        float
    """

    cumulative_returns = cumulative_returns.dropna()
    n_of_days = (cumulative_returns.index[-1] - cumulative_returns.index[0]
                ).days
    cagr =\
    (
        (        
        cumulative_returns.iloc[-1] 
        /
        cumulative_returns.iloc[0]
        ) ** (annual_days / n_of_days)
        - 1
    )
    return cagr

# print(f"The answer is {max(c_dct, key=c_dct.get).upper()} with {max(c_dct.values()):.2%}")

In [3]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['AMZN_ret_5'])
y = df[['AMZN_ret_5']]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)



def get_mse_train_test(model, X_train, Y_train, X_test, Y_test, name=None):
    """
    Compute MSE on train and test and store results in mse_results.
    If name is not provided, use model class name.
    """
    global mse_results
    y_train_pred = model.predict(X_train)
    mse_train = mean_squared_error(Y_train, y_train_pred)

    y_test_pred = model.predict(X_test)
    mse_test = mean_squared_error(Y_test, y_test_pred)

    model_name = name or model.__class__.__name__
    if model_name == "GridSearchCV":
        model_name = str(type(model.best_estimator_)).split(".")[-1].replace("'>", "")
    mse_results[str(model_name)] = {'mse_train': mse_train, 'mse_test': mse_test}

    print(f"Mean Squared Error (MSE) for {model_name}")
    print(f"MSE train: {mse_train:.6f}")
    print(f"MSE test: {mse_test:.6f}")

    return mse_train, mse_test

NameError: name 'df' is not defined

In [ ]:
from sklearn.metrics import mean_squared_error

mean_squared_error(y_true, y_pred)

In [1]:
def get_relationship(cols):
    rets = stock_data[cols].dropna()
    
    plt.scatter(rets[rets.columns[0]],rets[rets.columns[1]])
    r_min = rets.min().min()
    r_max = rets.max().max()

    np.linspace(r_min, r_max, 100)
    # rets.plot(kind='scatter')
    
    ln_x = np.linspace(r_min, r_max, 100)
    ln_y = np.linspace(r_min, r_max, 100)
    plt.plot(ln_x, ln_y, 'b--')
    
    corr_value = rets[cols[0]].corr(rets[cols[1]])
    print(f"Correlation between {cols[0]} and {cols[1]}: {corr_value:.4f}")
    plt.title(f"{cols[0]} vs {cols[1]}")
    plt.xlabel(cols[0])
    plt.ylabel(cols[1])
    plt.show()

# pair = ['XOM','GM']
# find_relationship(pair)

# Plot

In [21]:
# TODO get_plot lets-plot; basic 
from lets_plot import *
LetsPlot.setup_html()

wmt_melt = wmt.reset_index()
# wmt_melt = wmt_melt.rename(columns={'index':'Date'}). # optional depending if index is called 'Date'

p =\
(
    ggplot(wmt_melt, aes(x='Date')) +
    geom_line(aes(y='WMT'), color='grey', size=0.7) +
    geom_line(aes(y='sma_50'), color='orange', size=0.7,linetype=2) +   # amend sma as required
    geom_line(aes(y='sma_200'), color='blue', size=0.7,linetype=2) +
    geom_point(aes(y='sma_50'), data=wmt_melt[wmt_melt['trade']>0.0], color='red', size=3) +       # amend trade as required
    geom_point(aes(y='sma_50'), data=wmt_melt[wmt_melt['trade']<-0.0], color='blue', size=3) +
    ggtitle("Price Action with Signals") +
    ylab("Stock Price") +
    xlab("Date") +
    ggsize(1200, 500) 
)


p

NameError: name 'wmt' is not defined

In [22]:
# TODO get_plot matplotlib; secondary y  axis

ax =\
(
    ibm
    [["IBM", "sma_20", "sma_60", "trading_position"]]
    .plot(secondary_y = "trading_position",
          style = ["grey", "b--", "g--", "r"],
          figsize = [13, 7]
         )
)

ax.xaxis.grid(False)

(
    ax
    .legend(loc = "upper center",
            bbox_to_anchor = (0.55, 1.15),
            ncol = 4,
            shadow = True
           )
)

NameError: name 'ibm' is not defined

# Algos
## Linear Regression

In [5]:
import statsmodels.api as sm

# Ensure X_train is a DataFrame with column names
X_train_df = sm.add_constant(X_train)
X_test_df = sm.add_constant(X_test) # Add intercept

lr = sm.OLS(y_train, X_train_df).fit()
display(lr.summary())
get_mse_train_test(lr, 
                   X_train_df, 
                   y_train, 
                   X_test_df,
                   y_test, 
                   name="LR")

NameError: name 'pd' is not defined

## Elastic Net

In [ ]:
# ----------------------------------------------------------------------------
# 1. ELASTIC NET
# ----------------------------------------------------------------------------
from sklearn.linear_model import ElasticNet

elastic_net = ElasticNet(alpha=0.01, l1_ratio=0.1, max_iter=1000, random_state=42)
elastic_net.fit(X_train, y_train)
y_pred_en = elastic_net.predict(X_test)
print(f"Elastic Net - RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_en)):.4f}")
# Hyperparameter tuning
param_grid_en = {
    'alpha': [0.01, 0.1, 10.0],
    'l1_ratio': [0.1, 0.9]
}
grid_en = GridSearchCV(ElasticNet(max_iter=1000), param_grid_en, cv=5, scoring='neg_root_mean_squared_error')
grid_en.fit(X_train, y_train)
print(f"Best Elastic Net params: {grid_en.best_params_}")
# metrics
mse_train, mse_test = get_mse_train_test(elastic_net, X_train, y_train, X_test, y_test)

## Lasso

In [6]:
# ----------------------------------------------------------------------------
# 2. LASSO
# ----------------------------------------------------------------------------
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.01, max_iter=1000, random_state=42)
lasso.fit(X_train, y_train)
y_pred_lasso = lasso.predict(X_test)
print(f"LASSO - MSE: {mean_squared_error(y_test, y_pred_lasso):.4f}")

# Feature selection with LASSO
n_nonzero = np.sum(lasso.coef_ != 0)
# print(f"LASSO selected {n_nonzero} features out of {X_reg.shape[1]}")

mse_train, mse_test = get_mse_train_test(lasso, 
                                         X_train, 
                                         y_train, 
                                         X_test, 
                                         y_test)

NameError: name 'X_train' is not defined

## SVM

In [ ]:
# ----------------------------------------------------------------------------
# 3. SUPPORT VECTOR MACHINE (Regression)
# ----------------------------------------------------------------------------
from sklearn.svm import SVR

svr = SVR(kernel='rbf', C=1.0, epsilon=0.1)
svr.fit(X_train, y_train.values.ravel())
y_pred_svr = svr.predict(X_train)
print(f"SVR - RMSE: {np.sqrt(mean_squared_error(y_train.values.ravel(), y_pred_svr)):.4f}")

# Hyperparameter tuning
param_grid_svr = {
    'C': [0.1],
    'epsilon': [0.01, 0.1],
    'kernel': ['linear']
}
grid_svr = GridSearchCV(SVR(), param_grid_svr, cv=5, scoring='neg_root_mean_squared_error')
grid_svr.fit(X_train, y_train.values.ravel())  # Uncomment to run
print(f"Best SVR params: {grid_svr.best_params_}")

mse_train, mse_test = get_mse_train_test(grid_svr, 
                                         X_train, 
                                         y_train.values.ravel(), 
                                         X_test, 
                                         y_test.values.ravel())

## KNN

In [ ]:
# ----------------------------------------------------------------------------
# 4. K-NEAREST NEIGHBOR (Regression)
# ----------------------------------------------------------------------------
from sklearn.neighbors import KNeighborsRegressor

knn_reg = KNeighborsRegressor(n_neighbors=5, weights='uniform')
knn_reg.fit(X_train, y_train)
y_pred_knn = knn_reg.predict(X_train)
print(f"KNN Regressor - RMSE: {np.sqrt(mean_squared_error(y_train, y_pred_knn)):.4f}")

# Find optimal k
param_grid_knn = {'n_neighbors': range(1, 21)}
grid_knn = GridSearchCV(KNeighborsRegressor(), param_grid_knn, cv=5, scoring='neg_root_mean_squared_error')
grid_knn.fit(X_train, y_train)
print(f"Best k: {grid_knn.best_params_['n_neighbors']}")

mse_train, mse_test = get_mse_train_test(grid_knn, 
                                         X_train, 
                                         y_train, 
                                         X_test, 
                                         y_test)

## ARIMA


In [ ]:
# ============================================================================
# 10. ARIMA (Time Series)
# ============================================================================
from statsmodels.tsa.arima.model import ARIMA
import statsmodels.tsa.arima.model as stats
import warnings
warnings.filterwarnings('ignore')

baseline_ARIMA =\
(
    stats
    .ARIMA(endog = y_train,
           exog = X_train,
           order = [1, 0, 0]
          )
)
arima = baseline_ARIMA.fit()
[key for key in mse_results.keys() ]
y_pred_arima = arima.predict(
    start = 0,
    end = X.shape[0] -1,
    exog = X_test
)

# y_pred_arima
mse_train = \
(
    mean_squared_error(y_train,
                       y_pred_arima[:len(y_train)])
)
mse_test = \
(
    mean_squared_error(y_test,
                       y_pred_arima[-len(y_test):])
)
name = getattr(arima, 'name', None) or arima.__class__.__name__
mse_results[name] = {'mse_train': mse_train, 'mse_test': mse_test}b

## Decistion Tree

In [ ]:
# ----------------------------------------------------------------------------
# 5. DECISION TREE
# ----------------------------------------------------------------------------
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(max_depth=10, min_samples_split=5, random_state=42)
dt.fit(X_train, Y_train)
y_pred_dt = dt.predict(X_test)
print(f"Decision Tree - MSE: {mean_squared_error(Y_test, y_pred_dt):.4f}")

# Feature importance
feature_importance_dt = pd.DataFrame({
    'feature': [f'feature_{i}' for i in range(X_train.shape[1])],
    'importance': dt.feature_importances_
}).sort_values('importance', ascending=False)
print("Top 5 important features (DT):")
print(feature_importance_dt.head())


## Extra Trees

In [7]:
# ----------------------------------------------------------------------------
# 6. EXTRA TREES
# ----------------------------------------------------------------------------
from sklearn.ensemble import ExtraTreesRegressor

et = ExtraTreesRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
et.fit(X_train, Y_train)
y_pred_et = et.predict(X_test)
print(f"Extra Trees - RMSE: {np.sqrt(mean_squared_error(Y_test, y_pred_et)):.4f}")

mse_train, mse_test = get_mse_train_test(et, 
                                         X_train, 
                                         Y_train, 
                                         X_test, 
                                         Y_test)

NameError: name 'X_train' is not defined

## Random Forest

In [ ]:
# ----------------------------------------------------------------------------
# 7. RANDOM FOREST
# ----------------------------------------------------------------------------
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, 
       y_train)
y_pred_rf = rf.predict(X_test)
print(f"Random Forest - RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_rf)):.4f}")

# Hyperparameter tuning (commented out - uncomment to run)
param_grid_rf = {
    'n_estimators': [50, 200],
    'max_depth': [5, 10],
    'min_samples_split': [5, 10]
}
grid_rf = GridSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1), 
                       param_grid_rf, cv=3, scoring='neg_root_mean_squared_error')
grid_rf.fit(X_train, y_train)
print(f"Best RF params: {grid_rf.best_params_}")

mse_train, mse_test = get_mse_train_test(grid_rf, 
                                         X_train, 
                                         y_train, 
                                         X_test, 
                                         y_test)

## K means

In [ ]:
# K-means Clustering - Silhouette Score Method
## get_kmeans clustering
# ----------------------------------------------------------------------------
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

"""
TODO: ENSURE DATA IS SCALED BEFORE CLUSTERING
"""
# Silhouette score method
silhouette_ls = []
max_loop = len(X)

for k in range(2, max_loop):
        kmeans = KMeans(n_clusters = k,  
                        random_state = 627, 
                        n_init = 10)
        kmeans.fit(X)        
        silhouette_ls.append(silhouette_score(X, kmeans.labels_, random_state = 627)
                               )
        
fig = plt.figure(figsize=(16, 10)
                )

plt.plot(range(2, max_loop), silhouette_ls)

plt.xticks([i for i in range(2, max_loop)], 
           rotation=75)

plt.grid(True)

In [ ]:
"""
Elbow method and pick optimal k based on silhouette
"""
from sklearn import cluster 

# Apply K-Means with optimal k
optimal_k = 4   # TODO find optimal k to replace this value

# Fit with k-means
k_means = cluster.KMeans(n_clusters=optimal_k)
k_means.fit(X)

# plot 
## Visualize clusters
centroids = k_means.cluster_centers_

fig = plt.figure(figsize=(16,10)
                )

ax = fig.add_subplot(111)

scatter =\
(    
    ax
    .scatter(X.iloc[ : ,0], 
             X.iloc[ : ,1], 
             c = k_means.labels_, 
             cmap = "rainbow", 
             label = X.index)
)

ax.set_title("k-Means results")
ax.set_xlabel("Average Return")
ax.set_ylabel("Volatility")

plt.colorbar(scatter)

plt.plot(centroids[:,0], 
         centroids[:,1], 
         "sg", 
         markersize = 15, 
         color = "black")

# PLEASE NOTE THAT ALL THE RETURNS AND VOLS IN THIS PLOT ARE SCALED
##########################################

NameError: name 'X' is not defined

In [ ]:
print("    Cluster assignments:")
cluster_df = pd.DataFrame({'ticker': returns_df.columns, 'cluster': k_means.labels_})
for cluster_id in sorted(cluster_df['cluster'].unique()):
    tickers = cluster_df[cluster_df['cluster'] == cluster_id]['ticker'].tolist()
    print(f"    - Cluster {cluster_id}: {tickers}")

## PCA

In [ ]:
# ----------------------------------------------------------------------------
# PCA get_pca
# ----------------------------------------------------------------------------
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA


x = returns_df
RANDOM_SEED=42
NumEigenValues = 10
NumComponents = 4
x_train_raw, x_test_raw = train_test_split(x, test_size=0.2, random_state=RANDOM_SEED, shuffle=False)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train_raw) 
x_test = scaler.transform(x_test_raw)       # keep raw for actual resutls later

pca = PCA()
PrincipalComponent = pca.fit(x_train)

def PCWeights(pca_):

    weights = pd.DataFrame()

    for i in range(len(pca_.components_)
                  ):
        # w = pca_.components_[i] - pca_.components_[i].mean()
        # w = w / sum(abs(w))
        weights["weights_{}".format(i)] = pca_.components_[i] / sum(pca_.components_[i])

    weights = weights.values.T
    return weights # Team, be careful with indentation    
    # return w # Team, be careful with indentation


weights = PCWeights(pca) 


topPortfolios =\
(
    pd
    .DataFrame(pca.components_[ : NumComponents],
               columns = x.columns)
)

eigen_portfolios =\
(
    topPortfolios
    .div(topPortfolios.sum(axis=1), # normalising step
         axis = 0)
)

eigen_portfolios.index = [f"Portfolio {i}" for i in range(NumComponents)
                         ]
np.sqrt(pca.explained_variance_)

In [ ]:
display(
    pd
    .Series(np
           .cumsum(pca
                   .explained_variance_ratio_)
           )
    .to_frame("Explained Variance")
    .head(NumEigenValues)
    .style
    .format("{:,.2%}".format)
)

# Scree plot with individual variance
pd.Series(pca.explained_variance_ratio_).plot()
plt.title('Explained Variance')
plt.show()


# Scree plot with cumulative variance
pd.Series(np\
        .cumsum(pca
                .explained_variance_ratio_)
        ).plot()
plt.title('Cum Explained Variance')
plt.show()



In [ ]:
def calculate_sharpe_ratio(ts_returns, periods_per_year = 252):

    n_years = ts_returns.shape[0] / periods_per_year

    annualized_return = np.power(np.prod(1 + ts_returns), (1 / n_years)
                                ) - 1

    annualized_vol = ts_returns.std() * np.sqrt(periods_per_year)

    annualized_sharpe = annualized_return / annualized_vol

    return annualized_return, annualized_vol, annualized_sharpe

def valid_backtest_PCA_porfolios(eigen):
    
    eigen_prtfi =\
        (
            pd
            .DataFrame(data = {"weights": eigen.squeeze()
                              },
                       index = stock_tickers)
        )

    # Sanity Check: Ensure the order of tickers in X_Test_Raw matches the order in eigen
    if not list(eigen_prtfi.index) == stock_tickers: 
        raise ValueError("Sanity check failed: Mismatch in number of tickers between X_Test_Raw and eigen.")
    else:
        print("Prof. Roh's Message: 'Sanity check succeeded :)' The order of tickers in X_Test_Raw matches the order in eigen.")
    
    # Let's directly compute the dot product without sorting
    eigen_prtfi_returns =\
    (
        np
        .dot(X_Test_Raw, eigen)
    )
    
    eigen_portfolio_returns =\
    (
        pd
        .Series(eigen_prtfi_returns.squeeze(),
                index = x_test_index)
    )

    returns, vol, sharpe = calculate_sharpe_ratio(eigen_portfolio_returns)

    print("Our PCA-based Portfolio:\nReturn = %.2f%%\nVolatility = %.2f%%\nSharpe = %.2f"  %
          (returns * 100, vol * 100, sharpe)
         )

    # Compared with what? Equal-weightage Portfolio

    equal_weight_return =\
    (
        X_Test_Raw * (1 / len(pca.components_)
                     )
    ).sum(axis = 1)

    df_plot =\
        (
            pd
            .DataFrame({"ML Portfolio Return": eigen_portfolio_returns,
                        "Equal Weight Index": equal_weight_return},
                      index = x_test_index
                      )
        )

    (
        np
        .cumprod(df_plot + 1)
        .plot(title = "Returns of the equal weighted index vs. Eigen-Portfolio",
              figsize = [16, 8]
             )
    )

    plt.show()
    
    return returns, vol, sharpe

In [ ]:
stock_tickers = x.columns.to_list()
X_Test_Raw = x_test_raw
x_test_index = x.iloc[len(x_train):].index


res_ls = []
for idx in range(4):
    rets, vol, sharpe = valid_backtest_PCA_porfolios(eigen = weights[idx])
    res_ls.append((rets, vol, sharpe))

In [ ]:
# plot weights of PCs
for weight in weights[:4]:
    pd.Series(weight,index=returns_df.columns).plot(kind='bar',figsize=[24,4])
    plt.show()

In [ ]:
# ----------------------------------------------------------------------------
# agnes get_agnes agglo
# ----------------------------------------------------------------------------


# prepare annual returns
ann_rets = stock_data.pct_change().mean() * 252
# prepare volatiltiy
vol = stock_data.pct_change().std(ddof=1) * np.sqrt(252)

df = \
(
    pd.DataFrame({'return': ann_rets, 'volatility': vol})
)

# hierarchical cluster
# scale data
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
data = scaler.fit_transform(df)

df = pd.DataFrame(data, columns=df.columns, index=df.index)

X = df.copy()

# model fit
from scipy.cluster.hierarchy import dendrogram, linkage, ward
#Calulate linkage
Z = linkage(X, 
            method = "ward")
# Plot Dendogram

plt.figure(figsize=(18, 10)
          )
plt.title("Stocks Dendograms")

dendrogram(Z, labels = X.index)

plt.show()

from sklearn.cluster import AgglomerativeClustering
# from sklearn.cluster import KMeans, AgglomerativeClustering,AffinityPropagation, DBSCAN
from scipy.cluster.hierarchy import fcluster
# from scipy.cluster.hierarchy import dendrogram, linkage, cophenet
# from scipy.spatial.distance import pdist
# from sklearn.metrics import adjusted_mutual_info_score
# from sklearn import cluster, covariance, manifold

# choose a cluster from the longest vertical line
distance_threshold = 7

clusters = fcluster(Z, distance_threshold, criterion='distance')

chosen_clusters = pd.DataFrame(data=clusters, 
                               columns=['cluster']
                              )
display(chosen_clusters['cluster'].unique())

# do the transformation to assign clusters
# agglomarative clustering
nclust = 3

hc = AgglomerativeClustering(n_clusters = nclust, 
                             linkage = "ward")

clust_labels1 = hc.fit_predict(X)

In [ ]:
#visualisation
fig = plt.figure(figsize=[16,10]
)

ax = fig.add_subplot(111)

scatter = ax.scatter(X.iloc[:,0], 
                     X.iloc[:,1], 
                     c = clust_labels1, 
                     cmap = "rainbow")

ax.set_title("Hierarchical")
ax.set_xlabel("Average Return")
ax.set_ylabel("Volatility")

plt.colorbar(scatter)

## t-SNE (TSNE)

In [ ]:
# ----------------------------------------------------------------------------
# t-SNE get_tsne (is not clustering but dimension reduction for visualization only)
# ----------------------------------------------------------------------------
from sklearn.manifold import TSNE

k = 0

# prepare annual returns
ann_rets = stock_data.pct_change().mean() * 252
ann_vols = stock_data.pct_change().std() * np.sqrt(252)
df = \
(
    pd.DataFrame({'return': ann_rets, 'volatility': ann_vols})
)

# fit the clusters you want
X = df[['return', 'volatility']].values
kmeans = KMeans(n_clusters = k,  
                random_state = 627, 
                n_init = 10)
kmeans.fit(X)      
df['cluster'] = kmeans.labels_

# Fit t-SNE (2D projection)
tsne = TSNE(n_components=2, perplexity=8, random_state=42)
tsne_results = tsne.fit_transform(X)

# Add results to df
df['tsne_1'] = tsne_results[:, 0]
df['tsne_2'] = tsne_results[:, 1]

# Visualize t-SNE with cluster labels
plt.figure(figsize=(8,6))
scatter = plt.scatter(
    df['tsne_1'],
    df['tsne_2'],
    c=df['cluster'],
    cmap='tab10',
    s=80,
    alpha=0.8
)

# optional: show cluster centroids in tsne space
for c in sorted(df['cluster'].unique()):
    x = df[df['cluster']==c]['tsne_1'].mean()
    y = df[df['cluster']==c]['tsne_2'].mean()
    plt.text(x, y, str(c), fontsize=12, weight='bold')

plt.title('t-SNE: K-Means clusters')
plt.show()


# multiple features problems

In [ ]:
target = 'NFLX'
stock_tickers = [
    "NFLX", "AMZN", "AAPL", "META", "GOOGL"    # Stocks
]
index_tickers = [
    "^GSPC", "^DJI", "^VIX"        # Indices
]
ccy_tickers = [
    "JPY=X", "GBPUSD=X"           # FX rates
]
extract_tickers = stock_tickers + index_tickers + ccy_tickers

stock_data =\
(
    yf.download(
        extract_tickers,
        start = dt.datetime(2013, 1, 7),
        auto_adjust=False
    )
)['Adj Close']
# remove nulls
stock_data = stock_data.ffill()

[*********************100%***********************]  10 of 10 completed


In [ ]:
sub_df_list = []

# 5 day lags for all stocks
df_rets = stock_data[[ticker for ticker in stock_tickers if ticker != target]].copy() # remove target
lag = 5
df_rets = df_rets.pct_change(lag).shift(1).rename(columns={f'{ticker}':f'{ticker}_ret_{lag}' for ticker in stock_tickers})

sub_df_list.append(df_rets)

In [ ]:
# get ccy 
sub_df_list.append(stock_data[ccy_tickers].copy())
# get indices
sub_df_list.append(stock_data[index_tickers].copy())

In [ ]:

# lagged five-day, 15-day, 30-day, and 60-day returns of NFLX.
lag_list = [5, 15, 30, 60]
df_rets = stock_data[[target]].copy()

for lag in lag_list:
    df_rets[f'{target}_ret_{lag}'] = df_rets[target].pct_change(lag).shift(1)
    
df_rets = df_rets.drop(columns=target)

sub_df_list.append(df_rets)

In [ ]:
# get sma
def get_sma(df, sma_list):
    """
    Take DF and given list of SMA to enrich df when stock price is in first column
    """
    df = df.copy()
    for window in sma_list:
        df[f"sma_{window}"] =\
        (
            df[df.columns[0]]
            .rolling(window = window,
                     min_periods = int(window * .8))
            .mean()
        )

    return df

sma_list = [21, 63, 252]
df_sma = get_sma(stock_data[[target]].copy(), sma_list).drop(columns=target)

sub_df_list.append(df_sma)



In [ ]:
# get EMA
def get_ema(df, ema_list):
    """
    Take DF and given list of EMA to enrich df when stock price is in first column
    """
    df = df.copy()
    for window in ema_list:
        df[f"ema_{window}"] =\
        (
            df[df.columns[0]]
            .ewm(span = window)
            .mean()
        )

    return df

ema_list = [10, 30, 200]
df_ema = get_ema(stock_data[[target]].copy(), ema_list).drop(columns=target)

sub_df_list.append(df_ema)

In [ ]:
# get RSI
def get_rsi(df, period_list):
    """
    Calculate RSI (Relative Strength Index)
    Parameters:
    df: pandas df of closing price
    period_list: RSI period (default [10, 30, 200])
    Returns:
    pandas df with RSI values
    """
    # Make a copy to avoid SettingWithCopyWarning
    df = df.copy()
    
    # Calculate price changes
    delta = df[df.columns[0]].diff()
    
    for period in period_list:
        # Separate gains and losses
        gain = delta.clip(lower=0)
        loss = -delta.clip(upper=0)
        
        # get first average
        avg_gain = pd.Series(index=df.index, dtype='float64')
        avg_loss = pd.Series(index=df.index, dtype='float64')

        # get first average
        avg_gain.iloc[period] = gain.iloc[1:period+1].mean()
        avg_loss.iloc[period] = loss.iloc[1:period+1].mean()

        # Calculate rolling averages
        for i in range(period + 1, len(df)): 
            avg_gain.iloc[i] = (avg_gain.iloc[i-1] * (period - 1) + gain.iloc[i]) / period
            avg_loss.iloc[i] = (avg_loss.iloc[i-1] * (period - 1) + loss.iloc[i]) / period

        # Calculate RS (Relative Strength)
        rs = avg_gain / avg_loss
        
        # Calculate RSI
        rsi = 100 - (100 / (1 + rs))

        df[f'rsi_{period}'] = rsi
        
    return df

rsi_list = [10, 30, 200]
df_rsi = get_rsi(stock_data[[target]].copy(), rsi_list).drop(columns=target)

sub_df_list.append(df_rsi)

In [ ]:
# get_stochastic_oscillator
def get_stochastic_oscillator(ticker, start, end, period_list=[10, 30, 200]):
    """
    Get the stochastic oscillator for a given stock.
    """
    def STOK(close, low, high, n):
        STOK = ((close - low.rolling(n).min()) / \
                (high.rolling(n).max() - low.rolling(n).min()
                )
            ) * 100
        return STOK

    def STOD(close, low, high, n):
        STOK = ((close - low.rolling(n).min()) / \
                (high.rolling(n).max() - low.rolling(n).min()
                )
            ) * 100
        
        STOD = STOK.rolling(3).mean()
        return STOD

    df = yf.download(
                        ticker,
                        start=start,
                        end=end
                    ).droplevel('Ticker',axis=1)

    for period in period_list:
        df[f"%K{period}"] = STOK(df["Close"], df["Low"], df["High"], period)
        df[f"%D{period}"] = STOD(df["Close"], df["Low"], df["High"], period)

    return df.drop(columns=['Open', 'High', 'Low', 'Close', 'Volume'])

kd_list=[10, 30, 200]
df_kd = get_stochastic_oscillator(target, stock_data.index[0], stock_data.index[-1], kd_list)

sub_df_list.append(df_kd)

/var/folders/h2/r7qn2m9n1zb6y_0q191gdqth0000gn/T/ipykernel_41274/1249423668.py:22: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(
[*********************100%***********************]  1 of 1 completed


In [ ]:
# Rate of Change (RoC)
def get_roc(df, period_list):
    """
    Calculate the Rate of Change (RoC) for a given DataFrame and period.
    """
    df = df.copy()
    for period in period_list:
        M = df[df.columns[0]].diff(period - 1)
        N = df[df.columns[0]].shift(period - 1)
        df[f"roc_{period}"] = (M / N) * 100

    return df

roc_list = [10, 30]
df_roc = get_roc(stock_data[[target]].copy(), [10, 30]).drop(columns=target)

sub_df_list.append(df_roc)

In [ ]:
# correlation of QQQ and SPY
df_corr = stock_data['SPY'].pct_change()\
                            .rolling(20)\
                            .corr(stock_data[target]\
                                .pct_change())\
                            .to_frame('spy-qqq_corr')
display(df_corr)
# 20 day rolling vol of QQQ
qqq_vol = (
            stock_data[target].pct_change()
                            .rolling(20)
                            .std() * np.sqrt(252)
            ).to_frame(f'{target}_vol')
display(qqq_vol)

In [ ]:
# get target forward returns with lag
# y parameter
def get_target_parameter(df, target, fwd=5, lag=1):
    df_fwd_rets = df[[target]].copy()
    df_fwd_rets[f"{target}_fwd_rets"] = df_fwd_rets.pct_change(fwd).shift(-fwd-lag) 
                            # get fwd rets, shift by same amount, only trade day after signal computed
    return df_fwd_rets.drop(columns=target)

df_fwd_rets = get_target_parameter(stock_data, target, fwd=5, lag=1)

sub_df_list.append(df_fwd_rets)

In [ ]:
# combine df
df_combined =\
(
    pd.concat(sub_df_list,axis=1)
)
# fill forward for NaN
df_combined =\
(
    df_combined.ffill().dropna()
)

In [ ]:
from sklearn.model_selection import train_test_split

# Prepare features and target
X = df_combined.drop(columns=[f'{target}_fwd_rets'])
y = df_combined[f'{target}_fwd_rets']  # Note: removed extra brackets for cleaner y

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=627, shuffle=False
)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Run multiple Algos

In [ ]:
# establish metrics 
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

mse_results = {}

def get_mse_train_test(model, X_train, y_train, X_test, y_test, name=None):
    """
    Compute MSE on train and test and store results in mse_results.
    If name is not provided, use model class name.
    """
    global mse_results
    y_train_pred = model.predict(X_train)
    mse_train = mean_squared_error(y_train, y_train_pred)
    r2_train = r2_score(y_train, y_train_pred)
    mae_train = mean_absolute_error(y_train, y_train_pred)

    y_test_pred = model.predict(X_test)
    mse_test = mean_squared_error(y_test, y_test_pred)
    r2_test = r2_score(y_test, y_test_pred)
    mae_test = mean_absolute_error(y_test, y_test_pred)

    model_name = name or model.__class__.__name__
    if model_name == "GridSearchCV":
        model_name = str(type(model.best_estimator_)).split(".")[-1].replace("'>", "")
    

    mse_results[str(model_name)] = {
                                    'mse_train': mse_train, 
                                    'mse_test': mse_test,
                                    'r2_train': r2_train,
                                    'r2_test': r2_test,
                                    'mae_train': mae_train,
                                    'mae_test': mae_test
                                }

    print(f"Mean Squared Error (MSE) for {model_name}")
    print(f"MSE train: {mse_train:.6f}")
    print(f"MSE test: {mse_test:.6f}")
    print(f"R2 train: {r2_train:.6f}")
    print(f"R2 test: {r2_test:.6f}")
    print(f"MAE train: {r2_train:.6f}")
    print(f"MAE test: {r2_test:.6f}")

    return mse_train, mse_test, r2_train, r2_test, mae_train, mae_test

In [ ]:
# instantiate models
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor

# Initialize models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha = 10.0),
    'Lasso': Lasso(alpha = 0.01),
    'Elastic Net': ElasticNet(alpha = 0.01, l1_ratio = 0.5),
    'KNN': KNeighborsRegressor(n_neighbors = 10),
    'Decision Tree': DecisionTreeRegressor(max_depth = 5, random_state = 627),
    'Random Forest': RandomForestRegressor(n_estimators = 100, max_depth = 10, random_state = 627),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators = 100, max_depth = 3, random_state = 627)
}

for model_name, model in models.items():
    fit_model = model.fit(X_train_scaled, y_train)
    get_mse_train_test(fit_model, 
                   X_train_scaled, 
                   y_train, 
                   X_test_scaled,
                   y_test, 
                   name=model_name)

In [ ]:
# get results DataFrame

## Create results dataframe
metrics = pd.DataFrame(mse_results).T.sort_values(by='mse_test')
metrics['mse_ratio'] = metrics['mse_test'] / metrics['mse_train']
metrics

## Visualization of model comparison
ax =\
(
    metrics[['mse_train', 'mse_test', 'mse_ratio']].plot(
        secondary_y = 'mse_ratio',
        kind='bar',
        title='Model MSE Comparison (lower better)',
        ylabel='test / train MSE Ratio',
        figsize=(12, 10)
    )
)

ax.xaxis.grid(False)

(
    ax
    .legend(loc = "upper left",
            bbox_to_anchor = (0.55, 1.15),
            ncol = 4,
            shadow = True
           )
)

In [ ]:

# for model_name, res_dct in mse_results.items():
#     print(model_name, ":")
#     for metric, value in res_dct.items():
#         print(f"{metric}: {value}")

# # Find best model with one line
# best_model = min(mse_results, key=lambda m: mse_results[m]['mse_test'])
# print(f"Best performing model (by Test MSE): {best_model}")
# print(f"Minimum Test MSE: {mse_results[best_model]['mse_test']:.4f}")

# # Overfitting analysis
# print("\nOverfitting Analysis:")
# for model_name, res_dct in mse_results.items():
#     diff = res_dct['mse_test'] - res_dct['mse_train']
#     print(f"{model_name}: Train-Test difference = {diff:.4f}")



In [ ]:
# Initialize models
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

models = {
    'Linear Regression': LinearRegression(),
    'LASSO': Lasso(alpha = 0.1),
    'Ridge': Ridge(alpha = 1.0),
    'Decision Tree': DecisionTreeRegressor(random_state = 627),
    'Random Forest': RandomForestRegressor(n_estimators = 100, random_state = 627)
}

# perform training and evaluation
for model_name, model in models.items():
    fit_model = model.fit(X_train_scaled, y_train)
    get_mse_train_test(fit_model, 
                   X_train_scaled, 
                   y_train, 
                   X_test_scaled,
                   y_test, 
                   name=model_name)

In [ ]:
### get results DataFrame
## Create results dataframe
metrics = pd.DataFrame(mse_results).T.sort_values(by='mse_test')
metrics['mse_ratio'] = metrics['mse_test'] / metrics['mse_train']
display(metrics)

## Visualization of model comparison
ax =\
(
    metrics[['mse_train', 'mse_test', 'mse_ratio']].plot(
        secondary_y = 'mse_ratio',
        kind='bar',
        title='Model MSE Comparison (lower better)',
        ylabel='test / train MSE Ratio',
        figsize=(12, 10)
    )
)

ax.xaxis.grid(False)

(
    ax
    .legend(loc = "upper left",
            bbox_to_anchor = (0.55, 1.15),
            ncol = 4,
            shadow = True
           )
)

In [ ]:
from sklearn.model_selection import GridSearchCV

# Print Statements

In [ ]:
stock_data = stock_data.loc['2016-01-01': '2022-01-01']

for ticker in stock_data.columns:
    print(f"For {ticker}:")
    print(f"missing vals: {stock_data[ticker].isnull().sum()}")
    print(f"start: {stock_data[ticker].index[0]}, end: {stock_data[ticker].index[-1]}")
    print(f"Total trading days: {stock_data[ticker].shape[0]}")


In [ ]:
# total PNL and CAGR
initial_capital = 100_000
print(f"    Total P&L for Stock X (AAPL) using Bollinger Band strategy: ${aapl.pnl.sum()}")
print(f"    Initial Capital: $100,000")
print(f"    Final Portfolio Value: ${100_000+aapl.pnl.sum():,.2f}")
print(f"    Return: {get_cagr(aapl['cum_strategy_returns']):.2%}")

for ticker, df in stock_dct.items():
    print(f"{ticker}")
    print(f"- Total P&L: ${df.pnl.sum():,.2f}")
    print(f"- Sharpe Ratio: {get_sharpe(df['strategy_returns']):.5f}")

NameError: name 'aapl' is not defined

In [13]:
# DRAWDOWNS, CAGR, CALMAR RATIO
aapl_dd = get_drawdowns_plot(aapl_momentum)

print(f"    Maximum Drawdown: {-aapl_dd.max_dd.min():.2%}")
print(f"    Longest Drawdown Period: {aapl_dd.duration.max():.0f} days")
print(f"    CAGR: {get_cagr(aapl_momentum.cum_strategy_returns):.2%}")
print(f"    Calmar Ratio: {get_cagr(aapl_momentum.cum_strategy_returns) /(-aapl_dd.max_dd.min()) :.5f}")

NameError: name 'get_drawdowns_plot' is not defined

In [8]:
# TRAIN TEST SPLIT INFO

print(f"    Training set shape: {X_train.shape}")
print(f"    Testing set shape: {X_test.shape}")
print(f"    Date range for training: {df.index[0]} to {df.index[len(X_train)-1]}")
print(f"    Date range for testing: {df.index[-len(X_test)]} to {df.index[-1]}")

NameError: name 'X_train' is not defined

In [ ]:
# PCA explained variance
print(f"    Number of components for 90% variance: {np.argmax(np.cumsum(pca.explained_variance_ratio_) > .9) + 1}")    
print(f"    Explained variance ratios (first 5 components):")
for i, var in enumerate(pca.explained_variance_ratio_[:5], start=1):
    print(f"    - PC{i}: {var:.2%}")
print(f"    Cumulative variance explained by first 3 components: {np.cumsum(pca.explained_variance_ratio_)[2]:.2%}")